<h1 style='font-weight:bold; color:#003366; text-align:center;'>Taller: EDA y visualizaciones</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 - 2026</h4>

<h2 style='font-weight:bold; color:#003366;'>Análisis de cadena de farmacias Vitalis</h2>

Vitalis es una cadena uruguaya de ocho farmacias. El directorio pidió un informe con una pregunta
concreta:

> *"Venimos facturando más que nunca, pero la sensación en el mostrador es que vendemos peor.
> ¿Qué está pasando y qué sucursal necesita atención?"*

Tenemos tres archivos:

| Archivo | Qué tiene |
|---|---|
| `ventas_farmacias.csv` | Una fila por operación: fecha, sucursal, categoría, canal, tipo de cliente, monto |
| `sucursales.csv` | Las ocho sucursales, con fecha de apertura y metros cuadrados |
| `costos_mensuales.csv` | Alquiler, sueldos, otros costos e inversión en publicidad, por sucursal y mes |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def ruta_datos(nombre_archivo):
    """Busca el archivo en las ubicaciones habituales del curso, incluida Colab."""
    candidatas = [
        Path("../Datos/taller_visualizacion") / nombre_archivo,
        Path("Datos/taller_visualizacion") / nombre_archivo,
        Path("/content/Datos/taller_visualizacion") / nombre_archivo,
        Path("/content") / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontro {nombre_archivo}.")


CARPETA_SALIDAS = Path("salidas")
CARPETA_SALIDAS.mkdir(exist_ok=True)

# Paleta institucional UM: dos azules, un dorado para destacar y un gris para el resto.
AZUL = "#11224D"
AZUL_CLARO = "#6699CC"
DORADO = "#D4AF37"
GRIS = "#BBBBBB"

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.axisbelow"] = True

In [ ]:
ventas = pd.read_csv(ruta_datos("ventas_farmacias.csv"), parse_dates=["fecha"])
sucursales = pd.read_csv(ruta_datos("sucursales.csv"), parse_dates=["fecha_apertura"])
costos = pd.read_csv(ruta_datos("costos_mensuales.csv"))

# Una sola union al principio, para no repetirla en cada gráfico.
ventas = ventas.merge(sucursales[["codigo_sucursal", "sucursal"]], on="codigo_sucursal")
ventas["mes"] = ventas["fecha"].dt.to_period("M").dt.to_timestamp()
ventas["anio"] = ventas["fecha"].dt.year

FECHA_CORTE = ventas["fecha"].max()
MES_PARCIAL = FECHA_CORTE.to_period("M").to_timestamp()

print(f"{len(ventas):,} operaciones de {ventas['sucursal'].nunique()} sucursales")
print(f"Periodo: {ventas['fecha'].min():%d/%m/%Y} a {FECHA_CORTE:%d/%m/%Y}")
ventas.head(3)

<h2 style='font-weight:bold; color:#003366;'>1. Análisis de Gráficos </h2>

Para cada gráfico:

1. Votación a mano alzada: **apto para el informe** / **no apto**.
2. Quien votó "no apto" dice **cuál es el defecto en una frase**.
3. Decidir cuál es **el arreglo mínimo**: lo más chico que haya que cambiar para que sirva.

![Eje truncado](graficos_taller/01_antes.png)

![Torta](graficos_taller/02_antes.png)

![Orden alfabetico](graficos_taller/03_antes.png)

![Plato de fideos](graficos_taller/04_antes.png)

![Doble eje](graficos_taller/05_antes.png)

![Mes incompleto](graficos_taller/06_antes.png)

<h2 style='font-weight:bold; color:#003366;'>2. Las siete decisiones de un gráfico de entrega</h2>

El mismo gráfico, con los mismos datos, puede ser un borrador o una pieza de informe. La
diferencia son siete decisiones, y ninguna es de código:

| # | Decisión | La pregunta que se contesta |
|---|---|---|
| 1 | **La pregunta** | ¿Qué pregunta de negocio contesta este gráfico? Si no hay pregunta, no hay gráfico |
| 2 | **El tipo** | Evolución = línea. Comparación entre categorías = barras. Distribución = histograma o boxplot. Relación = dispersión. Composición = barras apiladas al 100% |
| 3 | **El orden** | Tiempo = cronológico. Categorías = por valor, salvo que haya un orden natural |
| 4 | **La escala** | Barras desde cero. Líneas pueden no arrancar en cero, pero entonces se avisa |
| 5 | **El foco** | Un color para lo que importa, gris para el contexto. Si todo está resaltado, nada lo está |
| 6 | **El texto** | Título que comunica rápidamente qué se está graficando. Unidad en el eje. Fecha de corte y fuente |
| 7 | **La conclusión** | La frase que va debajo. Si no se puede escribir, el gráfico no encontró nada |

<h3 style='font-weight:bold; color:#003366;'>La frase de conclusión</h3>

Lo que convierte un notebook en un análisis de negocio son las frases, no los gráficos. La
plantilla tiene tres partes y entra en dos renglones:

> **[Qué pasa] + [cuánto, con el número] + [qué habría que hacer o mirar].**

Tres ejemplos del mismo gráfico:

- *"Se observa que la facturación varía entre sucursales."* 
- *"Pocitos factura 7,3 millones y Salto 3,6 millones."* 
- *"Pocitos y Centro concentran el 35% de la facturación. Una caída de 10% en cualquiera de las dos pesa más que cerrar Salto entero, así que el seguimiento semanal debería empezar por ahí."* 

Ojo que algo importante a la hora de sacar conclusiones también es **lo que el gráfico NO permite concluir:**

- *"No permite concluir que Salto sea ineficiente: factura menos porque atiende una plaza
más chica. Para eso habría que mirar facturación por metro cuadrado o por empleado."*

<h2 style='font-weight:bold; color:#003366;'>El promedio de ventas baja sin que ninguna categoría caiga</h2>

El directorio dijo que *vendemos peor*. 

Lo primero a mirar es la venta promedio de la cadena:

In [ ]:
venta_cadena = (
    ventas[ventas["anio"].isin([2025, 2026])]
    .groupby("anio")["monto"]
    .agg(["mean", "median", "count"])
    .round(0)
)

display(venta_cadena)

variacion = venta_cadena.loc[2026, "median"] / venta_cadena.loc[2025, "median"] - 1
print(f"La venta mediana de la cadena varió {variacion:.1%} entre 2025 y 2026.")

Acá **la mediana baja igual que el promedio**. Eso ya descarta la explicación fácil:
si fuera culpa de unos pocos valores extremos, la mediana no se movería.

**Consigna:** Completá la celda para visualizar la **venta mediana por categoría** en
2025 y en 2026, y la variación entre los dos años. 

In [ ]:
dos_anios = ventas[ventas["anio"].isin([2025, 2026])]

venta_por_categoria = dos_anios.pivot_table(
    index="categoria", columns="anio", values="monto", aggfunc="median"
).round(0)

venta_por_categoria["variacion"] = (
    venta_por_categoria[2026] / venta_por_categoria[2025] - 1
)

## Mostrá venta mediana por categoría ordenada por su variación, de mayor a menor.

In [ ]:
# Y ahora el gráfico: ¿bajó alguna?
venta_por_categoria = dos_anios.pivot_table(
    index="categoria", columns="anio", values="monto", aggfunc="median"
).round(0)
venta_por_categoria["variacion"] = venta_por_categoria[2026] / venta_por_categoria[2025] - 1
orden = venta_por_categoria.sort_values("variacion")

fig, ax = plt.subplots(figsize=(9, 4.5))
barras = ax.barh(orden.index, orden["variacion"] * 100, color=AZUL_CLARO)
ax.bar_label(barras, fmt=" %.1f%%", fontsize=10)
ax.axvline(variacion * 100, color=DORADO, linewidth=2, linestyle="--")
ax.text(
    variacion * 100 + 0.3,
    len(orden) - 0.4,
    f"Toda la cadena: {variacion:.1%}",
    color="#7a6212",
    ha="left",
    va="center",
    fontsize=10,
    fontweight="bold",
)
ax.set_title(
    "Venta mediana por categoría: variación entre 2025 y 2026",
    loc="left",
    fontweight="bold",
)
ax.set_xlabel("Variación la venta mediana entre 2025 y 2026 (%)")
ax.set_xlim(variacion * 100 - 1.0, orden["variacion"].max() * 100 + 1.5)
ax.set_ylim(-0.7, len(orden) - 0.1)
fig.tight_layout()
plt.show()

**Consigna:** El gráfico que explica el total es el de **composición**. Completá el título.

In [ ]:
# Participación de cada categoría en la cantidad de operaciones, mes a mes.
mezcla = (
    ventas[ventas["mes"] < MES_PARCIAL]
    .pivot_table(index="mes", columns="categoria", values="id_ticket", aggfunc="count")
    .fillna(0)
)

mezcla_pct = mezcla.div(mezcla.sum(axis=1), axis=0)

orden_categorias = ["Medicamentos", "Perfumeria", "Dermocosmetica",
                    "Alimentos", "Accesorios", "Ortopedia"]

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.stackplot(
    mezcla_pct.index,
    [mezcla_pct[c] * 100 for c in orden_categorias],
    labels=orden_categorias,
    colors=[AZUL, AZUL_CLARO, DORADO, "#aa8fd9", "#9fb5b4", "#EAEAEA"],
)
ax.legend(loc="upper center", ncol=3, fontsize=9, frameon=False)
ax.set_title(
    "Completa el título del gráfico con un texto que explique qué se está mostrando",
    loc="left", fontweight="bold",
)
ax.set_ylabel("% de las operaciones del mes")
ax.set_ylim(0, 135)
fig.tight_layout()
plt.show()

In [ ]:
venta_por_categoria = dos_anios.pivot_table(index="categoria", columns="anio", values="monto", aggfunc="median"
).reset_index().round(0)

venta_por_categoria.sort_values(2026, ascending=False)

<h2 style='font-weight:bold; color:#003366;'>Qué sucursal necesita atención</h2>

Para responder esta pregunta lo primero que miramos es 

In [ ]:
ranking_2026 = (
    ventas[ventas["anio"] == 2026]
    .groupby("sucursal")["monto"]
    .sum()
    .div(1e6)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 4.5))
barras = ax.barh(ranking_2026.index, ranking_2026.values, color=AZUL_CLARO)
ax.bar_label(barras, fmt=" %.1f M", fontsize=10)
ax.set_title("Facturación 2026 por sucursal", loc="left")
ax.set_xlabel("Millones de $")
ax.set_xlim(0, ranking_2026.max() * 1.15)
fig.tight_layout()
plt.show()

In [ ]:
display(sucursales[["sucursal", "departamento", "fecha_apertura", "metros_cuadrados"]])

**Consigna:** Rehacé el ranking de forma que las ocho sucursales sean comparables.

Correcciones:

1. **La base.** Ciudad de la Costa no estuvo abierta todo 2026: hay que dividir por los días que
   estuvo abierta, no por el año entero.
2. **El corte.** El período termina a mitad de mes, así que el mes parcial entra igual para
   todas, pero conviene decirlo en el gráfico.



In [ ]:
inicio_2026 = pd.Timestamp("2026-01-01")

base = (
    ventas[ventas["anio"] == 2026]
    .groupby("codigo_sucursal")["monto"]
    .sum()
    .reset_index()
    .merge(sucursales, on="codigo_sucursal")
)

base["dias_abiertos"] = (FECHA_CORTE - base["fecha_apertura"].clip(lower=inicio_2026)).dt.days + 1

# Calculá facturación por día abierto, en miles de pesos
base["por_dia"] = 

# Graficar barras horizontales ordenadas por "por_dia" (completar x y y en el gráfico)

fig, ax = plt.subplots(figsize=(9, 4.5))
barras = ax.barh(x, y, color=AZUL_CLARO)
ax.bar_label(barras, fmt=" %.1f", fontsize=10)
ax.set_title(
    "Facturación promedio por día abierto de enero a 14 octubre de 2026",
    loc="left",
    fontweight="bold",
)
ax.set_xlabel("Miles de $ por día abierto")
ax.set_xlim(0, base["por_dia"].max() * 1.15)
ax.text(
    0.99, 0.05,
    f"Datos al {FECHA_CORTE:%d/%m/%Y}. Ciudad de la Costa abrió el 02/03/2026.",
    transform=ax.transAxes, ha="right", fontsize=8, color="#666666",
)
fig.tight_layout()
plt.show()


<h2 style='font-weight:bold; color:#003366;'>Qué tipo de productos generan más devoluciones </h2>

El archivo tiene una columna `devuelto` con 1 y 0, así que la tasa sale en una línea.

In [ ]:
tasa_mensual = (
    ventas[ventas["mes"] < MES_PARCIAL]
    .pivot_table(index="mes", columns="categoria", values="devuelto", aggfunc="mean")
    .mul(100)
)

fig, ax = plt.subplots(figsize=(10, 4.8))
for categoria in tasa_mensual.columns:
    ax.plot(tasa_mensual.index, tasa_mensual[categoria], marker="o", markersize=3, label=categoria)
ax.legend(ncol=3, fontsize=9, frameon=False)
ax.set_title("Tasa de devolución mensual por categoría", loc="left")
ax.set_ylabel("% de operaciones devueltas")
fig.tight_layout()
plt.show()

Una de las líneas pega saltos enormes de un mes al otro. Esto implica que esa categoría tenga un problema de calidad que va y viene, o que
hay otra cosa?

**Consigna:** cuántas operaciones hay detrás de
cada punto.

In [ ]:
# Contá las operaciones por mes y categoría (cambiar la función de agregación a "count")
operaciones = ## Completar

display(operaciones.median().round(0).sort_values().rename("operaciones por mes (mediana)"))

### Gráfico de barras horizontales con la tasa de devolución por categoría, ordenadas de menor a mayor.
resumen = (
    ventas.groupby("categoria")
    .agg(operaciones=("devuelto", "size"), devoluciones=("devuelto", "sum"))
    .assign(tasa=lambda d: d["devoluciones"] / d["operaciones"] * 100)
    .sort_values("tasa")
)

colores = [DORADO if c == "Ortopedia" else AZUL_CLARO for c in resumen.index]

fig, ax = plt.subplots(figsize=(9, 4.5))
barras = ax.barh(resumen.index, resumen["tasa"], color=colores)
# La tasa y su denominador van juntos: una tasa sola no se puede interpretar.
etiquetas = [
    f"  {fila['tasa']:.1f}%   (n = {fila['operaciones']:,.0f})" for _, fila in resumen.iterrows()
]
ax.bar_label(barras, labels=etiquetas, fontsize=9)
ax.set_title(
    "Tasa de devolución por categoría",
    loc="left", fontweight="bold",
)
ax.set_xlabel("% de operaciones devueltas, todo el período")
ax.set_xlim(0, resumen["tasa"].max() * 1.45)
fig.tight_layout()
plt.show()